In [1]:
%gui qt6
%matplotlib widget
import mne
import os
from data import eeg
from analysis import isc
from analysis import plotting
import numpy as np
from pathlib import Path
from ipywidgets import Layout, widgets
from IPython.display import display
from matplotlib import pyplot as plt
import pandas as pd
from scipy.stats import pearsonr


working_dir = Path(os.getcwd())
if working_dir.name == "notebooks":
    working_dir = working_dir.parent

print("Working directory:", working_dir)

data_dir = Path(os.getenv("EEG_WORK_DIR", "./out"))
if not data_dir.is_absolute():
    data_dir = working_dir / data_dir


Working directory: /home/zeyus/Projects/byd-hyperscanning-analysis


In [2]:
plt.ioff()
all_data = eeg.load_all_eeg(data_dir, preprocessed=True)


Loading BangBangYouAreDead: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 11.61it/s]


# Data Exploration

In [3]:
all_data["BangBangYouAreDead"][1]


<Raw | 72_HS101_BangBangYouAreDead_preprocessed_eeg.fif, 12 x 311752 (1247.0 s), ~28.6 MiB, data loaded>

In [4]:
# ── Widgets ────────────────────────────────────────────────────────────────────
initial_stimulus = eeg.STIMULI[1]
initial_subject_id = eeg.SUBJECT_IDS[0]
initial_raw = all_data[initial_stimulus][initial_subject_id]
initial_duration = initial_raw.n_times / initial_raw.info["sfreq"]
freq = initial_raw.info["sfreq"]

stim_select = widgets.Dropdown(
    options=[(s, s) for s in eeg.STIMULI],
    description="Stimulus:",
    value=initial_stimulus,
    layout=Layout(width="auto"),
)

subject_select = widgets.Dropdown(
    options=[(f"Subject {sid:02d}", sid) for sid in eeg.SUBJECT_IDS],
    description="Subject:",
    value=initial_subject_id,
    layout=Layout(width="auto"),
)

channel_selection = widgets.SelectMultiple(
    options=initial_raw.ch_names,
    value=["Cz", "P3", "C3"],
    description="Channels:",
    rows=12,
    layout=Layout(width="auto"),
)

time_start = widgets.BoundedFloatText(
    value=20.0,
    min=0.0,
    max=initial_duration,
    step=1.0,
    description="From (s):",
    style={"description_width": "initial"},
    layout=Layout(width="180px"),
)

time_end = widgets.BoundedFloatText(
    value=100.0,
    min=0.0,
    max=initial_duration,
    step=1.0,
    description="To (s):",
    style={"description_width": "initial"},
    layout=Layout(width="180px"),
)

plot_button = widgets.Button(
    description="Plot EEG",
    button_style="primary",
    icon="eye",
    layout=Layout(width="150px", height="36px"),
)

eeg_output = widgets.Output()

# ── Layout ─────────────────────────────────────────────────────────────────────
left_panel = widgets.VBox([stim_select, subject_select, channel_selection])
right_panel = widgets.VBox(
    [
        widgets.HBox([time_start, time_end]),
        plot_button,
    ]
)
controls = widgets.HBox(
    [left_panel, widgets.Box(layout=Layout(width="30px")), right_panel]
)

display(controls, eeg_output)


# ── Callback ───────────────────────────────────────────────────────────────────
def plot_eeg_data(_):
    eeg_output.clear_output(wait=True)
    with eeg_output:
        stimulus = stim_select.value
        subject_id = subject_select.value
        selected_channels = list(channel_selection.value)
        t_start = time_start.value
        t_end = 0
        if time_end.value > initial_duration:
            t_end = initial_duration
        elif time_end.value == 0:
            t_end = initial_duration
        else:
            t_end = time_end.value

        if t_end <= t_start:
            print("'To' must be greater than 'From'.")
            return

        raw = all_data[stimulus][subject_id]

        print(f"Showing data for Subject {subject_id:02d} - {stimulus}")
        print(
            f"Sample rate: {raw.info['sfreq']} Hz, Total duration: {raw.n_times / raw.info['sfreq']:.2f} seconds"
        )
        print(f"Time window: {t_start:.2f} - {t_end:.2f} seconds")

        picks = mne.pick_channels(raw.info["ch_names"], include=selected_channels)

        __ = raw.plot(
            start=t_start,
            duration=t_end - t_start,
            picks=picks,
            use_opengl=True,
            block=True,
            show_scrollbars=False,
            overview_mode="hidden",
            splash=False,
            verbose="error",
            show=False,
        )
        plt.show(block=True)
        plt.close("all")

        __ = raw.compute_psd(
            picks=picks,
            tmin=t_start,
            tmax=t_end,
            verbose="error",
            fmax=40.0,
            n_fft=int(freq * 5),
            n_overlap=int(freq),
            n_jobs=-1,
        ).plot(show=False)
        plt.show(block=True)
        plt.close("all")

        __ = raw.compute_psd(
            picks=picks,
            tmin=t_start,
            tmax=t_end,
            verbose="error",
            fmax=40.0,
            n_jobs=-1,
        ).plot(show=False)
        plt.show(block=True)
        plt.close("all")


plot_button.on_click(plot_eeg_data)


Output()

# Inter-Subject Correlation (ISC) Analysis

Pick the stimulus, subjects and time windows to include in ISC, then the ISC parameters.

In [12]:
from IPython.display import clear_output as _clear_cell_output

plt.ioff()


isc_results = {}
_pipeline_flags = {"isc": False, "apply": False, "isc_corr": False, "ts_corr": False}

FEATURE_LABELS = {
    "mean_lum": "Mean Luminance",
    "ebu_r128_M": "Loudness (LUFS, EBU R128)",
    "min_lum": "Min Luminance",
    "max_lum": "Max Luminance",
    "diff_lum": "Luminance Diff",
    "amp_rms": "Audio RMS (dBFS)",
    "amp_peak": "Audio Peak (dBFS)",
}

# ── Shared widgets (same objects referenced by all downstream cells) ─────────
shared_stim = widgets.Dropdown(
    options=[(s, s) for s in eeg.STIMULI],
    value=eeg.STIMULI[1],
    description="Stimulus:",
    layout=Layout(width="300px"),
)
shared_subjects = widgets.SelectMultiple(
    options=[(f"Subject {sid:02d}", sid) for sid in eeg.SUBJECT_IDS],
    value=list(eeg.SUBJECT_IDS),
    description="Subjects:",
    rows=len(eeg.SUBJECT_IDS),
    layout=Layout(width="200px"),
)
shared_window = widgets.FloatSlider(
    value=5.0,
    min=1.0,
    max=30.0,
    step=0.5,
    description="Window (s):",
    style={"description_width": "initial"},
    layout=Layout(width="400px"),
)
shared_step = widgets.FloatSlider(
    value=1.0,
    min=0.5,
    max=10.0,
    step=0.5,
    description="Step (s):",
    style={"description_width": "initial"},
    layout=Layout(width="400px"),
)
shared_n_comp = widgets.IntSlider(
    value=1,
    min=1,
    max=10,
    step=1,
    description="Components:",
    style={"description_width": "initial"},
    layout=Layout(width="400px"),
)
shared_tstart = widgets.BoundedFloatText(
    value=295.0,
    min=0.0,
    max=99999.0,
    step=1.0,
    description="Start (s):",
    style={"description_width": "initial"},
    layout=Layout(width="200px"),
)
shared_tend = widgets.BoundedFloatText(
    value=655.0,
    min=0.0,
    max=99999.0,
    step=1.0,
    description="End (s):",
    style={"description_width": "initial"},
    layout=Layout(width="200px"),
)
shared_component = widgets.BoundedIntText(
    value=1,
    min=1,
    max=10,
    step=1,
    description="Component:",
    style={"description_width": "initial"},
    layout=Layout(width="180px"),
)

# ── Surrogate / chance-level widgets ─────────────────────────────────────────
shared_run_surr = widgets.Checkbox(
    value=True,
    description="Compute chance level",
    style={"description_width": "initial"},
    layout=Layout(width="220px"),
)
shared_n_perm = widgets.IntSlider(
    value=200,
    min=50,
    max=1000,
    step=50,
    description="Permutations:",
    style={"description_width": "initial"},
    layout=Layout(width="400px"),
)


def _toggle_perm_slider(change):
    shared_n_perm.disabled = not change["new"]


shared_run_surr.observe(_toggle_perm_slider, names="value")

_time_range_note = widgets.Label("Leave End at 0 to use the full timeseries.")
_stale_label = widgets.HTML(value="")

run_button = widgets.Button(
    description="Run ISC",
    button_style="primary",
    icon="play",
    layout=Layout(width="150px", height="36px"),
)
isc_output = widgets.Output()

left_panel = widgets.VBox([shared_stim, shared_subjects])
right_panel = widgets.VBox(
    [
        shared_window,
        shared_step,
        shared_n_comp,
        widgets.HBox([shared_tstart, shared_tend]),
        _time_range_note,
        widgets.HBox([shared_run_surr, shared_n_perm]),
        widgets.HBox([run_button, _stale_label]),
    ]
)
controls = widgets.HBox(
    [left_panel, widgets.Box(layout=Layout(width="30px")), right_panel]
)
_clear_cell_output(wait=False)  # prevent duplicate displays on re-run
display(controls, isc_output)


# ── Cascade helpers ──────────────────────────────────────────────────────────
def _cascade_apply():
    if _pipeline_flags["apply"]:
        apply_weights(None)


def _cascade_isc_corr():
    if _pipeline_flags["isc_corr"]:
        correlate_features(None)


def _cascade_ts_corr():
    if _pipeline_flags["ts_corr"]:
        correlate_timecourses(None)


# ── Observers ────────────────────────────────────────────────────────────────
def _mark_isc_stale(change=None):
    if _pipeline_flags["isc"]:
        _stale_label.value = (
            '<span style="color:orange;font-size:12px;margin-left:8px;">'
            "&#9888; ISC stale \u2013 click Run ISC</span>"
        )


for _w in [
    shared_stim,
    shared_tstart,
    shared_tend,
    shared_window,
    shared_step,
    shared_n_comp,
]:
    _w.observe(_mark_isc_stale, names="value")


def _on_component_change(change):
    _cascade_apply()


shared_component.observe(_on_component_change, names="value")


# ── Component colour/width scheme (matches Poulsen et al. style) ─────────────
_COMP_COLORS = [
    "black",
    "#cc0000",
    "#1f77b4",
    "#2ca02c",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#7f7f7f",
    "#bcbd22",
    "#17becf",
]
_COMP_WIDTHS = [2.0, 1.4] + [1.1] * 8  # comp 1 thick black, comp 2 thinner red

# Context padding around a selected segment (seconds)
# _CONTEXT_PAD_S = 5.0
_CONTEXT_PAD_S = 0.0


# ── Run ISC ──────────────────────────────────────────────────────────────────
def run_isc(_):
    _stale_label.value = ""
    isc_output.clear_output(wait=True)
    with isc_output:
        stimulus = shared_stim.value
        subject_ids = list(shared_subjects.value)
        window_sec = shared_window.value
        step_sec = shared_step.value
        n_comp = shared_n_comp.value

        if len(subject_ids) < 2:
            print("Please select at least 2 subjects.")
            return

        arrays = [all_data[stimulus][s].get_data() for s in subject_ids]
        min_t = min(a.shape[1] for a in arrays)
        fs = int(all_data[stimulus][subject_ids[0]].info["sfreq"])
        full_dur_s = min_t / fs

        # ── User-specified segment ────────────────────────────────────────────
        t0_s = max(0.0, shared_tstart.value)
        t1_s = shared_tend.value if shared_tend.value > 0 else full_dur_s
        t0 = int(t0_s * fs)
        t1 = min(int(t1_s * fs), min_t)
        if t1 <= t0:
            print("End time must be greater than Start time.")
            return

        # ── Context padding: extend data slice if a sub-segment was chosen ───
        # Train CCA on the user segment; apply on the padded range so the
        # ISC timecourse extends _CONTEXT_PAD_S seconds beyond the boundaries.
        pad_samp = int(_CONTEXT_PAD_S * fs)
        t0_ext = max(0, t0 - pad_samp)
        t1_ext = min(min_t, t1 + pad_samp)
        t0_ext_s = t0_ext / fs
        t1_ext_s = t1_ext / fs
        # Only show context dashed lines when the segment doesn't cover the full data
        segment_selected = (t0_ext < t0) or (t1_ext > t1)

        # Training data: user-specified segment only
        data_array = np.array([a[:, t0:t1] for a in arrays])
        # Application data: padded range (may equal data_array if no padding)
        data_array_ext = (
            np.array([a[:, t0_ext:t1_ext] for a in arrays])
            if segment_selected
            else data_array
        )

        print(f"Stimulus  : {stimulus}")
        print(f"Subjects  : {subject_ids}  (N={len(subject_ids)})")
        print(
            f"Segment   : {t0_s:.1f}s \u2013 {t1_s:.1f}s  "
            f"({t1 - t0} samples @ {fs} Hz = {(t1 - t0) / fs:.1f}s)"
        )
        if segment_selected:
            print(
                f"Plot range: {t0_ext_s:.1f}s \u2013 {t1_ext_s:.1f}s  "
                f"(+{(t0 - t0_ext) / fs:.1f}s / +{(t1_ext - t1) / fs:.1f}s context)"
            )
        print(
            f"Window    : {window_sec}s  |  Step: {step_sec}s  |  Components: {n_comp}"
        )

        # Train on user segment, apply on (possibly padded) range
        W, ISC_train = isc.train_cca({stimulus: data_array})
        ISC, ISC_persecond, ISC_bysubject, A, window_times = isc.apply_cca(
            data_array_ext, W, fs, window_sec=window_sec, step_sec=step_sec
        )
        window_times = window_times + t0_ext_s  # absolute time offset
        n_comp = min(n_comp, ISC.shape[0])

        # Keep shared_component in bounds for the new result
        shared_component.max = n_comp
        if shared_component.value > n_comp:
            shared_component.value = n_comp

        # ── Surrogate chance-level estimation ────────────────────────────────
        chance_levels = None
        if shared_run_surr.value:
            n_perm = shared_n_perm.value
            print(
                f"\nEstimating chance level ({n_perm} circular-shift permutations, p < 0.01)..."
            )
            chance_levels = isc.compute_surrogate_chance_level(
                data_array_ext,
                W,
                fs,
                window_sec=window_sec,
                step_sec=step_sec,
                n_permutations=n_perm,
                p_threshold=0.01,
                n_comp=n_comp,
            )
            print(
                "Chance level range (p < 0.01): "
                + ", ".join(
                    f"Comp {c + 1}: [{chance_levels[c].min():.4f}, {chance_levels[c].max():.4f}]"
                    for c in range(n_comp)
                )
            )

        isc_results.update(
            {
                "W": W,
                "A": A,
                "ISC": ISC,
                "ISC_persecond": ISC_persecond,
                "ISC_bysubject": ISC_bysubject,
                "window_times": window_times,
                "stimulus": stimulus,
                "subject_ids": subject_ids,
                "fs": fs,
                "n_comp": n_comp,
                "info": all_data[stimulus][subject_ids[0]].info,
                "data_array": data_array_ext,
                "t0_s": t0_ext_s,
                "t1_s": t1_ext_s,
                "chance_levels": chance_levels,
            }
        )
        _pipeline_flags["isc"] = True

        # ── ISC timecourse plot (Poulsen et al. style) ───────────────────────
        window_times_min = window_times / 60.0

        fig, axes = plt.subplots(
            2, 1, figsize=(13, 5), gridspec_kw={"height_ratios": [3, 1]}
        )
        ax = axes[0]

        # Grey chance band
        if chance_levels is not None:
            for c in range(n_comp):
                ax.fill_between(
                    window_times_min,
                    -0.1,
                    chance_levels[c],
                    color="grey",
                    alpha=0.45,
                    label="p > 0.01, uncorrected" if c == 0 else None,
                    zorder=1,
                )

        # ISC component lines
        for c in range(n_comp):
            color = _COMP_COLORS[c] if c < len(_COMP_COLORS) else f"C{c}"
            lw = _COMP_WIDTHS[c] if c < len(_COMP_WIDTHS) else 1.0
            ax.plot(
                window_times_min,
                ISC_persecond[c],
                color=color,
                linewidth=lw,
                label=f"Comp {c + 1}  (ISC = {ISC[c]:.3f})",
                zorder=2 + c,
            )
            # save the component
            np.save('../in/isc_component1_byd_6min', ISC_persecond[c])

        ax.axhline(0, color="black", linewidth=0.5, linestyle="--", zorder=0)

        # Dashed vertical lines at the original segment boundaries
        if segment_selected:
            for boundary_s in [t0_s, t1_s]:
                ax.axvline(
                    boundary_s / 60.0,
                    color="black",
                    linewidth=1.0,
                    linestyle="--",
                    alpha=0.55,
                    zorder=5,
                )
        ymax = ISC_persecond[:n_comp].max() * 1.1
        ax.set_xlim(window_times_min[0], window_times_min[-1])
        ax.set_xticks(
            np.arange(
                np.ceil(window_times_min[0]), np.floor(window_times_min[-1]) + 1, 1.0
            ),
        )
        ax.set_ylim(bottom=-0.1, top=ymax)
        ax.set_xlabel("Time (min.)", fontsize=18)
        # increase y-tick and y-label font size
        ax.tick_params(axis="y", labelsize=18)
        ax.tick_params(axis="x", labelsize=18)
        ax.set_ylabel("ISC", fontsize=20)
        ax.set_yticks(np.arange(0, ymax, 0.1))
        ax.set_title(
            f"ISC per window \u2014 {stimulus}  "
            f"(N={len(subject_ids)}, window={window_sec}s, step={step_sec}s)"
        )
        ax.legend(loc="upper right", fontsize=12, frameon=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        # ── By-subject heatmap ───────────────────────────────────────────────
        heatmap_data = ISC_bysubject[:n_comp, :]
        abs_max = np.abs(heatmap_data).max()
        im = axes[1].imshow(
            heatmap_data, aspect="auto", cmap="RdBu_r", vmin=-abs_max, vmax=abs_max
        )
        axes[1].set_yticks(range(n_comp))
        axes[1].set_yticklabels([f"Comp {c + 1}" for c in range(n_comp)], fontsize=12)
        axes[1].set_xticks(range(len(subject_ids)))
        axes[1].set_xticklabels([f"S{sid:02d}" for sid in subject_ids], fontsize=12)
        axes[1].set_title(
            f"ISC by subject (leave-one-out pairwise, range [{-abs_max:.3f}, {abs_max:.3f}])"
        )
        plt.colorbar(im, ax=axes[1], orientation="vertical", fraction=0.02, pad=0.01)
        plt.tight_layout()
        plt.show(block=True)
    plt.close("all")

    # Cascade to downstream cells that have already been run
    _cascade_apply()
    _cascade_isc_corr()


# Clear any previously registered handlers before re-registering (prevents duplicate calls on cell re-run)
run_button._click_handlers.callbacks.clear()
run_button.on_click(run_isc)

# window corresponding to Parra paper.
# seg_start_s = (10*60 + 53) - 360  # 293s
# seg_end_s = 10*60 + 53            # 653s


Output()

## Component Inspection
Scalp topographies (from scalp projection matrix **A**) and spatial filter weights (**W**) for the components returned by CCA.

In [6]:
n_display_components = 3

n_components_slider = widgets.IntSlider(
    value=n_display_components,
    min=1,
    max=isc_results["n_comp"] if isc_results else 12,
    step=1,
    description="Components:",
    style={"description_width": "initial"},
    layout=Layout(width="400px"),
)

display(n_components_slider)

topo_button = widgets.Button(
    description="Inspect Components",
    button_style="info",
    icon="search",
    layout=Layout(width="200px", height="36px"),
)
topo_output = widgets.Output()

display(topo_button, topo_output)


def inspect_components(_):
    global n_display_components
    topo_output.clear_output(wait=True)

    with topo_output:
        if not isc_results:
            print("Run ISC first.")
            return

        A = isc_results["A"]
        W = isc_results["W"]
        n_comp = isc_results["n_comp"]
        info = isc_results["info"]
        ch_names = info["ch_names"]

        cmap = plotting.parula_map  # use if defined in the session
        n_rows = (n_display_components + 2) // 3
        n_cols = min(n_comp, n_display_components, 3)
        # ── Scalp topographies ─────────────────────────────────────────────────
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 3.5 * n_rows))
        if n_display_components == 1:
            axes = [axes]

        for comp_i in range(n_display_components):
            row = comp_i // 3
            col = comp_i % 3
            ax = axes[row, col] if n_display_components > 1 else axes[0]
            mne.viz.plot_topomap(
                A[:, comp_i],
                info,
                cmap=cmap,
                show=False,
                contours=6,
                axes=ax,
                sphere=0.09,
                extrapolate="head",
                vlim=(A[:, comp_i].min(), A[:, comp_i].max()),
            )
            ax.set_title(
                f"Comp {comp_i + 1}\nISC = {isc_results['ISC'][comp_i]:.3f}", fontsize=9
            )

        fig.suptitle(f"Scalp projections (A) — {isc_results['stimulus']}", fontsize=11)
        plt.tight_layout()
        plt.show()

        # ── Spatial filter weight matrix (W) ───────────────────────────────────
        # Normalize each column to unit norm — the absolute scale of CCA filters
        # is arbitrary; only the relative weights across channels matter
        W_display = W[:, :n_comp]
        W_display = W_display / np.linalg.norm(W_display, axis=0, keepdims=True)
        abs_max = np.abs(W_display).max()

        fig2, ax = plt.subplots(
            figsize=(max(5, n_comp * 0.9), max(4, len(ch_names) * 0.35))
        )
        im = ax.imshow(
            W_display, aspect="auto", cmap="RdBu_r", vmin=-abs_max, vmax=abs_max
        )
        ax.set_yticks(range(len(ch_names)))
        ax.set_yticklabels(ch_names, fontsize=8)
        ax.set_xticks(range(n_comp))
        ax.set_xticklabels([f"Comp {c + 1}" for c in range(n_comp)], fontsize=9)
        ax.set_title(
            "Spatial filters (W, unit-norm per component) — rows are channels",
            fontsize=10,
        )
        plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        plt.tight_layout()
        plt.show()
    # clear plot
    plt.close("all")


# update the value of the n_components_display
def update_n_display_components(change):
    global n_display_components
    n_display_components = change["new"]
    inspect_components(
        None
    )  # re-render the topographies with the new number of components


n_components_slider.observe(update_n_display_components, names="value")

topo_button.on_click(inspect_components)


IntSlider(value=3, description='Components:', layout=Layout(width='400px'), max=12, min=1, style=SliderStyle(d…

Button(button_style='info', description='Inspect Components', icon='search', layout=Layout(height='36px', widt…

Output()

## Stimulus–Brain Correlation
Load per-stimulus audiovisual features (mean luminance, LUFS loudness, and more) from the composite CSV and compute their time-matched Pearson correlation with each ISC component timecourse.

> **Requires** `Run ISC` to have been executed. Features are interpolated from the stimulus frame-rate to ISC window center times before correlating.

In [7]:
# FEATURE_LABELS, pd, and pearsonr are defined in the ISC cell above.

feature_select = widgets.SelectMultiple(
    options=list(FEATURE_LABELS.keys()),
    value=["mean_lum", "ebu_r128_M"],
    description="Features:",
    rows=len(FEATURE_LABELS),
    layout=Layout(width="260px"),
)

corr_button = widgets.Button(
    description="Correlate",
    button_style="warning",
    icon="bar-chart",
    layout=Layout(width="150px", height="36px"),
)

corr_output = widgets.Output()

display(widgets.HBox([feature_select, corr_button]), corr_output)


def correlate_features(_):
    corr_output.clear_output(wait=True)

    with corr_output:
        if not isc_results:
            print("Run ISC first.")
            return

        selected_features = list(feature_select.value)
        if not selected_features:
            print("Select at least one feature.")
            return

        stimulus = isc_results["stimulus"]
        window_times = isc_results["window_times"]
        ISC_persecond = isc_results["ISC_persecond"]
        n_comp = isc_results["n_comp"]

        csv_path = data_dir / f"{stimulus}_composite_frame_level_analysis.csv"
        if not csv_path.exists():
            print(f"Feature CSV not found:\n  {csv_path}")
            print("Run  src/composite-stimuli-features.py  to generate it first.")
            return

        feat_df = pd.read_csv(csv_path)
        feat_df = feat_df.replace([np.inf, -np.inf], np.nan)
        feat_df = feat_df.ffill().bfill()

        feat_interp: dict[str, np.ndarray] = {}
        for feat in selected_features:
            if feat not in feat_df.columns:
                print(f"  Feature '{feat}' not found in CSV \u2013 skipping.")
                continue
            # Interpolate feature values to match the ISC time points
            feat_interp[feat] = np.interp(
                window_times,
                feat_df["timestamp"].values,
                feat_df[feat].values,
            )

        if not feat_interp:
            print("No valid features found.")
            return

        n_feat = len(feat_interp)
        r_matrix = np.full((n_comp, n_feat), np.nan)
        p_matrix = np.full((n_comp, n_feat), np.nan)

        fig, axes = plt.subplots(
            n_comp,
            n_feat,
            figsize=(5.5 * n_feat, 3.2 * n_comp),
            squeeze=False,
        )

        for fi, (feat, fvals) in enumerate(feat_interp.items()):
            feat_label = FEATURE_LABELS.get(feat, feat)
            fvals_z = (fvals - fvals.mean()) / (fvals.std() + 1e-12)

            for ci in range(n_comp):
                ax = axes[ci, fi]
                isc_ts = ISC_persecond[ci]
                isc_z = (isc_ts - isc_ts.mean()) / (isc_ts.std() + 1e-12)

                r, p = pearsonr(isc_ts, fvals)
                r_matrix[ci, fi] = r
                p_matrix[ci, fi] = p

                ax2 = ax.twinx()
                ax.plot(
                    window_times,
                    isc_z,
                    color="#1f77b4",
                    linewidth=0.9,
                    label="ISC (z)",
                    alpha=0.85,
                )
                ax2.plot(
                    window_times,
                    fvals_z,
                    color="#d62728",
                    linewidth=0.9,
                    linestyle="--",
                    label=f"{feat_label} (z)",
                    alpha=0.75,
                )
                ax.axhline(0, color="gray", linewidth=0.4, linestyle=":")

                ax.set_ylabel("ISC (z-score)", color="#1f77b4", fontsize=8)
                ax2.set_ylabel(f"{feat_label} (z-score)", color="#d62728", fontsize=8)
                ax.set_xlabel("Time (s)", fontsize=8)

                sig = (
                    "***"
                    if p < 0.001
                    else "**"
                    if p < 0.01
                    else "*"
                    if p < 0.05
                    else "n.s."
                )
                ax.set_title(
                    f"Comp {ci + 1} \u00d7 {feat_label}\n"
                    f"r = {r:.3f}  p = {p:.4f}  {sig}",
                    fontsize=9,
                )
                ax.grid(True, alpha=0.2)
                lines1, labels1 = ax.get_legend_handles_labels()
                lines2, labels2 = ax2.get_legend_handles_labels()
                ax.legend(
                    lines1 + lines2, labels1 + labels2, fontsize=7, loc="upper right"
                )

        plt.suptitle(
            f"ISC \u2013 Feature Correlation \u2014 {stimulus}", fontsize=12, y=1.01
        )
        plt.tight_layout()
        plt.show()

        feat_labels = [FEATURE_LABELS.get(f, f) for f in feat_interp]
        abs_max = np.nanmax(np.abs(r_matrix))

        fig2, ax = plt.subplots(figsize=(max(3.5, n_feat * 1.8), max(2, n_comp * 0.9)))
        im = ax.imshow(
            r_matrix, aspect="auto", cmap="RdBu_r", vmin=-abs_max, vmax=abs_max
        )
        ax.set_xticks(range(n_feat))
        ax.set_xticklabels(feat_labels, rotation=30, ha="right", fontsize=9)
        ax.set_yticks(range(n_comp))
        ax.set_yticklabels([f"Comp {c + 1}" for c in range(n_comp)], fontsize=9)

        for ci in range(n_comp):
            for fi in range(n_feat):
                sig = (
                    "***"
                    if p_matrix[ci, fi] < 0.001
                    else "**"
                    if p_matrix[ci, fi] < 0.01
                    else "*"
                    if p_matrix[ci, fi] < 0.05
                    else ""
                )
                text_color = (
                    "white" if abs(r_matrix[ci, fi]) > abs_max * 0.6 else "black"
                )
                ax.text(
                    fi,
                    ci,
                    f"{r_matrix[ci, fi]:.3f}{sig}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=text_color,
                )

        ax.set_title(
            f"Pearson r \u2014 ISC components \u00d7 Stimulus Features\n{stimulus}",
            fontsize=10,
        )
        plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Pearson r")
        plt.tight_layout()
        plt.show()

        _pipeline_flags["isc_corr"] = True
    plt.close("all")


corr_button.on_click(correlate_features)


Output()

## Apply Spatial Filter
Project the EEG data through the selected CCA component to produce per-subject component timecourses.
The result is stored in `component_data` as a `(subjects × samples)` numpy array.

In [8]:
apply_button = widgets.Button(
    description="Apply & Plot",
    button_style="success",
    icon="check",
    layout=Layout(width="150px", height="36px"),
)
apply_output = widgets.Output()

# shared_component is defined in the ISC cell above; displaying it here
# means changes made in either cell are immediately reflected in both.
display(widgets.HBox([shared_component, apply_button]), apply_output)

component_data = None


def apply_weights(_):
    global component_data
    apply_output.clear_output(wait=True)
    with apply_output:
        if not isc_results:
            print("Run ISC first.")
            return

        comp_i = shared_component.value - 1
        W = isc_results["W"]
        data_array = isc_results["data_array"]
        subject_ids = isc_results["subject_ids"]
        fs = isc_results["fs"]
        stimulus = isc_results["stimulus"]
        t0_s = isc_results.get("t0_s", 0.0)

        spatial_filter = W[:, comp_i]
        component_data = np.array(
            [spatial_filter @ data_array[i] for i in range(len(subject_ids))]
        )

        T = component_data.shape[1]
        t = np.arange(T) / fs + t0_s

        print(
            f"Component {shared_component.value}  "
            f"(ISC = {isc_results['ISC'][comp_i]:.4f})"
        )
        print(f"Output shape : {component_data.shape}  (subjects \u00d7 samples)")
        print("Stored in    : component_data")

        fig, ax = plt.subplots(figsize=(13, 4))
        for i, sid in enumerate(subject_ids):
            ax.plot(t, component_data[i], linewidth=0.6, alpha=0.7, label=f"S{sid:02d}")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude (a.u.)")
        ax.set_title(
            f"Component {shared_component.value} timecourses \u2014 {stimulus}  "
            f"(ISC = {isc_results['ISC'][comp_i]:.3f})"
        )
        ax.legend(loc="upper right", fontsize=7, ncol=2)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        _pipeline_flags["apply"] = True
        _cascade_ts_corr()
    plt.close("all")


apply_button.on_click(apply_weights)


Output()

### Per-Subject Timecourse–Feature Correlation
Correlate each subject's spatially-filtered component timecourse (from **Apply Spatial Filter**) with the stimulus features at the full EEG sample rate. Shows per-subject and mean Pearson r.

In [9]:
# ── Settings panel (shared widgets displayed again for convenience) ───────────
# Changing shared_component here auto-cascades to Apply SF → this cell.
# Changing stimulus / time range here updates the ISC cell widgets too;
# click "Re-run ISC" to propagate those changes through the full pipeline.
_ts_rerun_isc_btn = widgets.Button(
    description="Re-run ISC",
    button_style="primary",
    icon="refresh",
    layout=Layout(width="130px", height="30px"),
)
_ts_rerun_isc_btn.on_click(run_isc)

_ts_settings_label = widgets.HTML(
    '<span style="font-size:12px;color:#555;">'
    "<b>Shared settings</b> &mdash; "
    "Component change auto-updates timecourse plots. "
    "Stimulus / time range changes require <i>Re-run ISC</i>."
    "</span>"
)

_ts_settings_panel = widgets.VBox(
    [
        _ts_settings_label,
        widgets.HBox([shared_stim, shared_component]),
        widgets.HBox([shared_tstart, shared_tend, _ts_rerun_isc_btn]),
    ]
)

ts_feature_select = widgets.SelectMultiple(
    options=list(FEATURE_LABELS.keys()),
    value=["mean_lum", "ebu_r128_M"],
    description="Features:",
    rows=len(FEATURE_LABELS),
    layout=Layout(width="260px"),
)

ts_corr_button = widgets.Button(
    description="Correlate",
    button_style="warning",
    icon="bar-chart",
    layout=Layout(width="150px", height="36px"),
)

ts_corr_output = widgets.Output()

display(
    _ts_settings_panel,
    widgets.HBox([ts_feature_select, ts_corr_button]),
    ts_corr_output,
)


def correlate_timecourses(_):
    ts_corr_output.clear_output(wait=True)
    with ts_corr_output:
        if component_data is None:
            print("Apply spatial filter first (click 'Apply & Plot' above).")
            return
        if not isc_results:
            print("Run ISC first.")
            return

        selected_features = list(ts_feature_select.value)
        if not selected_features:
            print("Select at least one feature.")
            return

        stimulus = isc_results["stimulus"]
        subject_ids = isc_results["subject_ids"]
        fs = isc_results["fs"]
        t0_s = isc_results.get("t0_s", 0.0)

        N, T = component_data.shape
        t_eeg = np.arange(T) / fs + t0_s

        csv_path = data_dir / f"{stimulus}_composite_frame_level_analysis.csv"
        if not csv_path.exists():
            print(f"Feature CSV not found:\n  {csv_path}")
            print("Run  src/composite-stimuli-features.py  to generate it first.")
            return

        feat_df = pd.read_csv(csv_path)
        feat_df = feat_df.replace([np.inf, -np.inf], np.nan)
        feat_df = feat_df.ffill().bfill()

        # ── Resample to the LOWER of the two rates ───────────────────────────
        # EEG component: ~250 Hz.  Features: ~30 fps.
        # Correct approach: downsample EEG to feature timestamps, not the reverse.
        # Upsampling features to 250 Hz would inflate N and yield spurious p-values.
        mask = (feat_df["timestamp"] >= t_eeg[0]) & (feat_df["timestamp"] <= t_eeg[-1])
        feat_df_win = feat_df[mask].reset_index(drop=True)
        feat_times = feat_df_win["timestamp"].values

        if len(feat_times) < 2:
            print("Not enough feature samples within the EEG time window.")
            return

        feat_fs_approx = 1.0 / np.median(np.diff(feat_times))
        print(f"EEG rate     : {fs} Hz  ({T} samples, {T / fs:.1f} s)")
        print(
            f"Feature rate : ~{feat_fs_approx:.2f} Hz  ({len(feat_times)} frames in window)"
        )
        print(
            f"Correlation  : EEG downsampled to feature timestamps (~{feat_fs_approx:.2f} Hz)"
        )

        # Downsample each subject's component timecourse to the feature timestamps
        comp_at_feat = np.array(
            [np.interp(feat_times, t_eeg, component_data[i]) for i in range(N)]
        )  # (N, n_feat_frames)
        mean_comp_at_feat = comp_at_feat.mean(axis=0)

        # Collect feature values at their native timestamps (no interpolation needed)
        feat_interp: dict[str, np.ndarray] = {}
        for feat in selected_features:
            if feat not in feat_df_win.columns:
                print(f"  Feature '{feat}' not found in CSV \u2013 skipping.")
                continue
            feat_interp[feat] = feat_df_win[feat].values

        if not feat_interp:
            print("No valid features found.")
            return

        n_feat = len(feat_interp)
        r_subj = {f: np.full(N, np.nan) for f in feat_interp}
        p_subj = {f: np.full(N, np.nan) for f in feat_interp}
        r_mean = {}
        p_mean = {}

        for feat, fvals in feat_interp.items():
            for i in range(N):
                r_subj[feat][i], p_subj[feat][i] = pearsonr(comp_at_feat[i], fvals)
            r_mean[feat], p_mean[feat] = pearsonr(mean_comp_at_feat, fvals)

        # ── Overlay: mean timecourse (downsampled) vs feature ───────────────
        # Full-res EEG is shown as a faint background for visual context only;
        # the bold line and all statistics use the downsampled signal.
        fig, axes = plt.subplots(n_feat, 1, figsize=(14, 3.2 * n_feat), squeeze=False)
        mean_z = (mean_comp_at_feat - mean_comp_at_feat.mean()) / (
            mean_comp_at_feat.std() + 1e-12
        )

        for fi, (feat, fvals) in enumerate(feat_interp.items()):
            feat_label = FEATURE_LABELS.get(feat, feat)
            fvals_z = (fvals - fvals.mean()) / (fvals.std() + 1e-12)
            ax = axes[fi, 0]
            ax2 = ax.twinx()

            # Faint full-res per-subject traces (visual context, not used for stats)
            for i in range(N):
                ts_full_z = (component_data[i] - component_data[i].mean()) / (
                    component_data[i].std() + 1e-12
                )
                ax.plot(t_eeg, ts_full_z, color="#1f77b4", linewidth=0.2, alpha=0.12)
            # Downsampled mean — this is what the correlation is computed on
            ax.plot(
                feat_times,
                mean_z,
                color="#1f77b4",
                linewidth=1.0,
                label=f"Mean component (z, ~{feat_fs_approx:.1f} Hz)",
                alpha=0.9,
            )
            ax2.plot(
                feat_times,
                fvals_z,
                color="#d62728",
                linewidth=0.9,
                linestyle="--",
                label=f"{feat_label} (z)",
                alpha=0.75,
            )
            ax.axhline(0, color="gray", linewidth=0.4, linestyle=":")

            ax.set_ylabel("Component (z-score)", color="#1f77b4", fontsize=8)
            ax2.set_ylabel(f"{feat_label} (z-score)", color="#d62728", fontsize=8)
            ax.set_xlabel("Time (s)", fontsize=8)

            r_m, p_m = r_mean[feat], p_mean[feat]
            sig_m = (
                "***"
                if p_m < 0.001
                else "**"
                if p_m < 0.01
                else "*"
                if p_m < 0.05
                else "n.s."
            )
            ax.set_title(
                f"Comp {shared_component.value} \u00d7 {feat_label}   "
                f"r(mean) = {r_m:.3f}  p = {p_m:.4f}  {sig_m}  |  "
                f"mean r(per-subj) = {np.mean(r_subj[feat]):.3f}"
                f"  [N={len(feat_times)} frames @ ~{feat_fs_approx:.1f} Hz]",
                fontsize=9,
            )
            ax.grid(True, alpha=0.2)
            lines1, labels1 = ax.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc="upper right")

        plt.suptitle(
            f"Component {shared_component.value} Timecourse \u00d7 Stimulus Features \u2014 {stimulus}",
            fontsize=12,
            y=1.01,
        )
        plt.tight_layout()
        plt.show()

        # ── Per-subject r bar chart ─────────────────────────────────────────
        subj_labels = [f"S{sid:02d}" for sid in subject_ids]
        fig2, axes2 = plt.subplots(
            1,
            n_feat,
            figsize=(max(4, 2.5 * n_feat), max(3, N * 0.45 + 1.5)),
            squeeze=False,
        )

        for fi, (feat, fvals) in enumerate(feat_interp.items()):
            feat_label = FEATURE_LABELS.get(feat, feat)
            ax = axes2[0, fi]
            rs = r_subj[feat]
            bar_colors = ["#4878d0" if r >= 0 else "#ee854a" for r in rs]
            bars = ax.barh(
                range(N),
                rs,
                color=bar_colors,
                alpha=0.8,
                edgecolor="white",
                linewidth=0.5,
            )
            r_m = r_mean[feat]
            ax.axvline(
                r_m,
                color="black",
                linewidth=1.5,
                linestyle="--",
                label=f"r(mean ts) = {r_m:.3f}",
            )
            ax.axvline(
                np.mean(rs),
                color="gray",
                linewidth=1.2,
                linestyle=":",
                label=f"mean of r = {np.mean(rs):.3f}",
            )
            ax.axvline(0, color="black", linewidth=0.6)
            ax.set_yticks(range(N))
            ax.set_yticklabels(subj_labels, fontsize=8)
            ax.set_xlabel("Pearson r", fontsize=9)
            ax.set_title(feat_label, fontsize=9)
            ax.legend(fontsize=7)
            ax.grid(True, alpha=0.25, axis="x")

            for i, (bar, r_val, p_val) in enumerate(zip(bars, rs, p_subj[feat])):
                sig = (
                    "***"
                    if p_val < 0.001
                    else "**"
                    if p_val < 0.01
                    else "*"
                    if p_val < 0.05
                    else ""
                )
                x_text = r_val + (0.01 if r_val >= 0 else -0.01)
                ax.text(
                    x_text,
                    i,
                    f"{r_val:.3f}{sig}",
                    va="center",
                    ha="left" if r_val >= 0 else "right",
                    fontsize=7,
                )

        plt.suptitle(
            f"Per-Subject Pearson r \u2014 Comp {shared_component.value} \u00d7 Features\n"
            f"{stimulus}  [~{feat_fs_approx:.1f} Hz, N={len(feat_times)} frames]",
            fontsize=11,
            y=1.02,
        )
        plt.tight_layout()
        plt.show()

        # ── Summary heatmap: subjects × features ───────────────────────────
        r_table = np.array([r_subj[f] for f in feat_interp]).T  # (N, n_feat)
        feat_labels = [FEATURE_LABELS.get(f, f) for f in feat_interp]
        abs_max = np.nanmax(np.abs(r_table))

        fig3, ax3 = plt.subplots(figsize=(max(3.5, n_feat * 1.8), max(2, N * 0.5)))
        im = ax3.imshow(
            r_table, aspect="auto", cmap="RdBu_r", vmin=-abs_max, vmax=abs_max
        )
        ax3.set_xticks(range(n_feat))
        ax3.set_xticklabels(feat_labels, rotation=30, ha="right", fontsize=9)
        ax3.set_yticks(range(N))
        ax3.set_yticklabels(subj_labels, fontsize=9)

        for si in range(N):
            for fi, feat in enumerate(feat_interp):
                r_val = r_table[si, fi]
                p_val = p_subj[feat][si]
                sig = (
                    "***"
                    if p_val < 0.001
                    else "**"
                    if p_val < 0.01
                    else "*"
                    if p_val < 0.05
                    else ""
                )
                text_color = "white" if abs(r_val) > abs_max * 0.6 else "black"
                ax3.text(
                    fi,
                    si,
                    f"{r_val:.3f}{sig}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=text_color,
                )

        ax3.set_title(
            f"Per-Subject Pearson r \u2014 Comp {shared_component.value} \u00d7 Features\n"
            f"{stimulus}  [~{feat_fs_approx:.1f} Hz, N={len(feat_times)} frames]",
            fontsize=10,
        )
        plt.colorbar(im, ax=ax3, fraction=0.03, pad=0.02, label="Pearson r")
        plt.tight_layout()
        plt.show()

        _pipeline_flags["ts_corr"] = True
    plt.close("all")


ts_corr_button.on_click(correlate_timecourses)


Output()

# Frequency Analysis

ISC -> on decomposed timeseries .

does a specific band influence ISC

- possibly use frequency-filtered source data AFTER running ISC using the originally computed weights

# question
- Online reference for EEG data?
- Does it make sense to use common reference given the channel density=


# Evoked Response
Compare component projections per subject for visual events
(scene change)
